# 07 — Podróż danych przez pipeline

**Cel:** prześledzenie jak **jeden plik STAR-Counts** jest przekształcany przez kolejne funkcje pipeline'u LUAD-HUBA — od surowego pliku z GDC do finalnej macierzy ekspresji.

Notebook pokazuje **ten sam fragment danych na każdym etapie**, podświetlając co dokładnie zmienia każda transformacja. Wszystkie operacje wykonywane są prawdziwymi funkcjami z `src/` na prawdziwym pliku STAR-Counts.

## Mapa transformacji

```
SUROWY PLIK .tsv (z GDC, GENCODE v36)
    │  ~60666 linii: komentarz #, nagłówek, 4 wiersze meta, 60660 genów
    │  9 kolumn (3 metryki zliczeń + 3 znormalizowane + metadane genu)
    │
    ▼  parse_star_counts()
    │     • pomija komentarze (#)
    │     • selekcja 9 wymaganych kolumn
    │     • usunięcie 4 wierszy meta (N_unmapped, N_multimapping, ...)
    │     • walidacja (ENSG, brak duplikatów)
    │     • rzutowanie zliczeń na Int64
    │
PARQUET (data/interim/star_counts/)
    │  60660 genów × 9 kolumn (czyste, otypowane)
    │
    ▼  build_expression_matrix()
    │     • mapowanie plik → sample_id (z sample sheet)
    │     • wybór JEDNEJ metryki (tpm_unstranded)
    │     • złączenie z innymi próbkami (geny × próbki)
    │     • filtr biotype → protein_coding
    │
MACIERZ EKSPRESJI (data/processed/expression_matrix.parquet)
       19962 genów × N próbek
       (ten plik = jedna kolumna nazwana sample_id)
```

---

## Konfiguracja

In [1]:
import sys
from pathlib import Path

import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.ingest.star_parser import (
    parse_star_counts,
    META_ROW_IDS,
    REQUIRED_COLUMNS,
    COUNT_COLUMNS,
)
from src.transform.expression_matrix import build_expression_matrix

# Konfiguracja wyświetlania polars
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(50)

# Plik przykładowy (jeden surowy STAR-Counts z kohorty)
DEMO_FILE = next((PROJECT_ROOT / "data" / "raw").rglob("*augmented_star_gene_counts.tsv"))
print(f"Plik przykładowy: {DEMO_FILE.name}")
print(f"Rozmiar: {DEMO_FILE.stat().st_size / 1024:.0f} KB")

# Geny które będziemy śledzić przez cały pipeline (markery LUAD + różne biotypy)
TRACKED_GENES = {
    "ENSG00000136352": "NKX2-1 (marker różnicowania, protein_coding)",
    "ENSG00000146648": "EGFR (onkogen, protein_coding)",
    "ENSG00000133703": "KRAS (onkogen, protein_coding)",
}

Plik przykładowy: 9ebc079c-069f-4468-8b9e-680e39a3c4ad.rna_seq.augmented_star_gene_counts.tsv
Rozmiar: 4145 KB


---
## Etap 0 — Surowy plik STAR-Counts

Tak wygląda plik bezpośrednio z GDC. Wczytujemy pierwsze linie **jako czysty tekst**, by zobaczyć strukturę zanim cokolwiek ją dotknie.

In [2]:
# Surowy tekst - pierwsze 11 linii
with open(DEMO_FILE, "r") as f:
    raw_lines = [next(f) for _ in range(11)]

print("=== SUROWY PLIK (pierwsze 11 linii, jako tekst) ===\n")
for i, line in enumerate(raw_lines):
    # Skracamy długie linie dla czytelności
    display = line.rstrip("\n")
    if len(display) > 95:
        display = display[:92] + "..."
    print(f"{i:2} | {display}")

=== SUROWY PLIK (pierwsze 11 linii, jako tekst) ===

 0 | # gene-model: GENCODE v36
 1 | gene_id	gene_name	gene_type	unstranded	stranded_first	stranded_second	tpm_unstranded	fpkm_un...
 2 | N_unmapped			2179752	2179752	2179752			
 3 | N_multimapping			8983540	8983540	8983540			
 4 | N_noFeature			1418566	41909136	41941975			
 5 | N_ambiguous			9515634	2356772	2359047			
 6 | ENSG00000000003.15	TSPAN6	protein_coding	7434	3692	3742	63.6498	22.2244	26.0120
 7 | ENSG00000000005.6	TNMD	protein_coding	0	0	0	0.0000	0.0000	0.0000
 8 | ENSG00000000419.13	DPM1	protein_coding	2968	1532	1436	95.5001	33.3455	39.0285
 9 | ENSG00000000457.14	SCYL3	protein_coding	555	530	513	3.1316	1.0934	1.2798
10 | ENSG00000000460.17	C1orf112	protein_coding	643	627	576	4.1830	1.4606	1.7095


**Co widzimy w surowym pliku:**

- **Linia 0:** komentarz `# gene-model: GENCODE v36` — poprzedzony `#`, do pominięcia
- **Linia 1:** nagłówek — 9 kolumn oddzielonych tabulatorami
- **Linie 2–5:** **wiersze meta** STAR (`N_unmapped`, `N_multimapping`, `N_noFeature`, `N_ambiguous`) — statystyki dopasowania, **nie geny**. Mają puste pola w kolumnach `gene_name`, `gene_type` i metrykach znormalizowanych
- **Linia 6+:** właściwe geny — `gene_id` to wersjonowany identyfikator Ensembl (`ENSG...15`)

### 9 kolumn pliku

| Kolumna | Znaczenie |
|---|---|
| `gene_id` | identyfikator Ensembl (wersjonowany) |
| `gene_name` | symbol genu (HGNC) |
| `gene_type` | biotyp GENCODE (protein_coding, lncRNA, ...) |
| `unstranded` | zliczenia surowe (niespecyficzne niciowo) |
| `stranded_first` / `stranded_second` | zliczenia specyficzne niciowo |
| `tpm_unstranded` | **TPM** — znormalizowane (transcripts per million) |
| `fpkm_unstranded` / `fpkm_uq_unstranded` | FPKM — znormalizowane |

In [3]:
# Surowy wiersz meta vs surowy wiersz genu - kontrast
print("=== Surowy wiersz META (N_unmapped) ===")
for line in raw_lines:
    if line.startswith("N_unmapped"):
        parts = line.rstrip("\n").split("\t")
        for col, val in zip(REQUIRED_COLUMNS, parts):
            print(f"  {col:22} = {val!r}")
        break

print("\n=== Surowy wiersz GENU (TSPAN6) - pierwszy gen w pliku ===")
for line in raw_lines:
    if line.startswith("ENSG"):
        parts = line.rstrip("\n").split("\t")
        for col, val in zip(REQUIRED_COLUMNS, parts):
            print(f"  {col:22} = {val!r}")
        break

=== Surowy wiersz META (N_unmapped) ===
  gene_id                = 'N_unmapped'
  gene_name              = ''
  gene_type              = ''
  unstranded             = '2179752'
  stranded_first         = '2179752'
  stranded_second        = '2179752'
  tpm_unstranded         = ''
  fpkm_unstranded        = ''
  fpkm_uq_unstranded     = ''

=== Surowy wiersz GENU (TSPAN6) - pierwszy gen w pliku ===
  gene_id                = 'ENSG00000000003.15'
  gene_name              = 'TSPAN6'
  gene_type              = 'protein_coding'
  unstranded             = '7434'
  stranded_first         = '3692'
  stranded_second        = '3742'
  tpm_unstranded         = '63.6498'
  fpkm_unstranded        = '22.2244'
  fpkm_uq_unstranded     = '26.0120'


Wiersz meta ma **puste stringi** (`''`) w `gene_name`, `gene_type` oraz w kolumnach TPM/FPKM. To dlatego, że meta to statystyki dopasowania, nie geny — nie mają biotypu ani znormalizowanej ekspresji. Te 4 wiersze musimy usunąć, bo zaburzyłyby analizę.

---
## Etap 1 — `parse_star_counts()`

Pierwsza funkcja transformująca. Wczytuje surowy TSV i wykonuje pięć operacji:

1. **Pomija komentarze** (`comment_prefix="#"`) — linia GENCODE znika
2. **Selekcja kolumn** — zostają tylko `REQUIRED_COLUMNS` (9 kolumn)
3. **Usunięcie wierszy meta** — `filter(~gene_id.is_in(META_ROW_IDS))`
4. **Walidacja** — wszystkie `gene_id` muszą zaczynać się od `ENSG`, brak duplikatów
5. **Rzutowanie typów** — kolumny zliczeń (`unstranded`, `stranded_first/second`) na `Int64`

Wykonujemy ją na naszym pliku:

In [4]:
# Wywołanie prawdziwej funkcji parse_star_counts
parsed = parse_star_counts(DEMO_FILE)

print("=== PO parse_star_counts ===")
print(f"Wymiary: {parsed.height} genów × {parsed.width} kolumn")
print(f"\nMETA_ROW_IDS (usuwane): {sorted(META_ROW_IDS)}")
print(f"\nTypy kolumn po parsowaniu:")
for col, dtype in zip(parsed.columns, parsed.dtypes):
    marker = "  <- zrzutowane na Int64" if col in COUNT_COLUMNS else ""
    print(f"  {col:22} {str(dtype):10}{marker}")

=== PO parse_star_counts ===
Wymiary: 60660 genów × 9 kolumn

META_ROW_IDS (usuwane): ['N_ambiguous', 'N_multimapping', 'N_noFeature', 'N_unmapped']

Typy kolumn po parsowaniu:
  gene_id                String    
  gene_name              String    
  gene_type              String    
  unstranded             Int64       <- zrzutowane na Int64
  stranded_first         Int64       <- zrzutowane na Int64
  stranded_second        Int64       <- zrzutowane na Int64
  tpm_unstranded         Float64   
  fpkm_unstranded        Float64   
  fpkm_uq_unstranded     Float64   


In [5]:
# Porównanie: ile wierszy zniknęło
n_raw_lines = sum(1 for _ in open(DEMO_FILE))
print("=== Bilans wierszy ===")
print(f"  Linii w surowym pliku:        {n_raw_lines}")
print(f"  - 1 komentarz (#)")
print(f"  - 1 nagłówek")
print(f"  - 4 wiersze meta")
print(f"  = genów po parsowaniu:        {parsed.height}")
print(f"\n  Sprawdzenie: {n_raw_lines} - 1 - 1 - 4 = {n_raw_lines - 6}")
assert parsed.height == n_raw_lines - 6, "Bilans się nie zgadza!"
print("  ✓ Bilans się zgadza")

print("\n=== Czy wiersze meta faktycznie zniknęły? ===")
meta_present = parsed.filter(pl.col("gene_id").is_in(META_ROW_IDS))
print(f"  Wierszy meta w sparsowanych danych: {meta_present.height} (oczekiwane 0)")

=== Bilans wierszy ===
  Linii w surowym pliku:        60666
  - 1 komentarz (#)
  - 1 nagłówek
  - 4 wiersze meta
  = genów po parsowaniu:        60660

  Sprawdzenie: 60666 - 1 - 1 - 4 = 60660
  ✓ Bilans się zgadza

=== Czy wiersze meta faktycznie zniknęły? ===
  Wierszy meta w sparsowanych danych: 0 (oczekiwane 0)


In [6]:
# Nasze śledzone geny PO parsowaniu
print("=== Śledzone geny po parse_star_counts ===\n")
for ensg, desc in TRACKED_GENES.items():
    row = parsed.filter(pl.col("gene_id").str.starts_with(ensg))
    if row.height > 0:
        r = row.row(0, named=True)
        print(f"{desc}")
        print(f"  gene_id={r['gene_id']}  unstranded={r['unstranded']} (Int64)  "
              f"tpm={r['tpm_unstranded']:.2f}\n")

=== Śledzone geny po parse_star_counts ===

NKX2-1 (marker różnicowania, protein_coding)
  gene_id=ENSG00000136352.19  unstranded=3389 (Int64)  tpm=41.59

EGFR (onkogen, protein_coding)
  gene_id=ENSG00000146648.19  unstranded=8502 (Int64)  tpm=26.38

KRAS (onkogen, protein_coding)
  gene_id=ENSG00000133703.13  unstranded=2542 (Int64)  tpm=14.42



**Co się zmieniło na Etapie 1:**

- Komentarz GENCODE i nagłówek — wchłonięte przez parser (nagłówek stał się nazwami kolumn)
- 4 wiersze meta — **usunięte** (60664 → 60660 wierszy danych)
- Kolumny zliczeń — teraz typu `Int64` zamiast string (gotowe do operacji liczbowych)
- Plik jest **czysty i otypowany**, ale wciąż ma wszystkie 9 kolumn i wszystkie 60660 genów

Na tym etapie pipeline zapisuje wynik jako **parquet** w `data/interim/star_counts/`. Każdy z 601 plików kohorty przechodzi przez tę samą funkcję.

---
## Etap 2 — Wybór metryki (wewnątrz `build_expression_matrix`)

Druga funkcja transformująca buduje macierz. Pierwszą rzeczą, którą robi z każdym plikiem, jest **wybór jednej metryki** spośród sześciu kolumn liczbowych.

Pipeline domyślnie używa **`tpm_unstranded`** (TPM — najlepsze do porównań międzypróbkowych, bo normalizuje długość genu i głębokość sekwencjonowania). Z 9 kolumn zostają **dwie**: `gene_id` + wybrana metryka.

Pokazujemy co dokładnie się dzieje — z 9 kolumn do gene_id + TPM:

In [7]:
METRIC = "tpm_unstranded"

# To co build_expression_matrix robi z pojedynczym plikiem: bierze gene_id + metrykę
single_column = parsed.select(["gene_id", METRIC])

print(f"=== Wybór metryki: {METRIC} ===")
print(f"Przed: {parsed.width} kolumn -> Po: {single_column.width} kolumny (gene_id + {METRIC})\n")
print("Odrzucone kolumny:")
for col in parsed.columns:
    if col not in ("gene_id", METRIC):
        print(f"  - {col}")

print(f"\n=== Fragment (gene_id + {METRIC}) ===")
print(single_column.head(8))

=== Wybór metryki: tpm_unstranded ===
Przed: 9 kolumn -> Po: 2 kolumny (gene_id + tpm_unstranded)

Odrzucone kolumny:
  - gene_name
  - gene_type
  - unstranded
  - stranded_first
  - stranded_second
  - fpkm_unstranded
  - fpkm_uq_unstranded

=== Fragment (gene_id + tpm_unstranded) ===
shape: (8, 2)
┌────────────────────┬────────────────┐
│ gene_id            ┆ tpm_unstranded │
│ ---                ┆ ---            │
│ str                ┆ f64            │
╞════════════════════╪════════════════╡
│ ENSG00000000003.15 ┆ 63.6498        │
│ ENSG00000000005.6  ┆ 0.0            │
│ ENSG00000000419.13 ┆ 95.5001        │
│ ENSG00000000457.14 ┆ 3.1316         │
│ ENSG00000000460.17 ┆ 4.183          │
│ ENSG00000000938.13 ┆ 24.6321        │
│ ENSG00000000971.16 ┆ 25.0933        │
│ ENSG00000001036.14 ┆ 67.504         │
└────────────────────┴────────────────┘


W macierzy nazwa kolumny metryki zostaje **zastąpiona przez `sample_id`** (identyfikator próbki z sample sheet) — bo w macierzy geny×próbki każda kolumna to jedna próbka. W tym demo nie mamy sample sheet dla pojedynczego pliku, więc pokazujemy mechanizm na pełnej kohorcie w Etapie 4.

---
## Etap 3 — Filtr biotype

Kolejna transformacja w `build_expression_matrix`: **filtr po `gene_type`**. Pipeline zachowuje tylko geny `protein_coding`, odrzucając lncRNA, pseudogeny, miRNA itd.

Powód: do analizy ekspresji i przeżywalności interesują nas geny kodujące białka. Pozostałe biotypy to ~67% wpisów, ale niosą inny rodzaj sygnału (lub szum).

In [11]:
# Rozkład biotypów w pliku
biotype_counts = (
    parsed.group_by("gene_type")
    .agg(pl.len().alias("liczba"))
    .sort("liczba", descending=True)
)
print("=== Rozkład biotypów (gene_type) w pliku ===")
print(biotype_counts.head(15))

n_total = parsed.height
n_protein = parsed.filter(pl.col("gene_type") == "protein_coding").height
print(f"\nprotein_coding: {n_protein} / {n_total} = {n_protein/n_total*100:.1f}%")

=== Rozkład biotypów (gene_type) w pliku ===
shape: (15, 2)
┌────────────────────────────────────┬────────┐
│ gene_type                          ┆ liczba │
│ ---                                ┆ ---    │
│ str                                ┆ u32    │
╞════════════════════════════════════╪════════╡
│ protein_coding                     ┆ 19962  │
│ lncRNA                             ┆ 16901  │
│ processed_pseudogene               ┆ 10167  │
│ unprocessed_pseudogene             ┆ 2614   │
│ misc_RNA                           ┆ 2212   │
│ snRNA                              ┆ 1901   │
│ miRNA                              ┆ 1881   │
│ TEC                                ┆ 1057   │
│ snoRNA                             ┆ 943    │
│ transcribed_unprocessed_pseudogene ┆ 939    │
│ transcribed_processed_pseudogene   ┆ 500    │
│ rRNA_pseudogene                    ┆ 497    │
│ IG_V_pseudogene                    ┆ 187    │
│ IG_V_gene                          ┆ 145    │
│ transcribed_unitary_pseudo

In [12]:
# Filtr biotype - co zostaje, co wypada
BIOTYPE = "protein_coding"

kept = parsed.filter(pl.col("gene_type") == BIOTYPE)
dropped = parsed.filter(pl.col("gene_type") != BIOTYPE)

print(f"=== Filtr biotype = '{BIOTYPE}' ===")
print(f"  Zachowane: {kept.height} genów")
print(f"  Odrzucone: {dropped.height} genów")
print(f"  60660 -> {kept.height}\n")

# Przykłady tego co WYPADA (różne biotypy)
print("=== Przykłady genów ODRZUCONYCH (nie protein_coding) ===")
examples_dropped = (
    dropped.filter(pl.col("gene_type").is_in(["lncRNA", "miRNA", "snRNA", "processed_pseudogene"]))
    .group_by("gene_type")
    .first()
    .select(["gene_id", "gene_name", "gene_type", "tpm_unstranded"])
)
print(examples_dropped)

=== Filtr biotype = 'protein_coding' ===
  Zachowane: 19962 genów
  Odrzucone: 40698 genów
  60660 -> 19962

=== Przykłady genów ODRZUCONYCH (nie protein_coding) ===
shape: (4, 4)
┌───────────────────┬───────────┬──────────────────────┬────────────────┐
│ gene_id           ┆ gene_name ┆ gene_type            ┆ tpm_unstranded │
│ ---               ┆ ---       ┆ ---                  ┆ ---            │
│ str               ┆ str       ┆ str                  ┆ f64            │
╞═══════════════════╪═══════════╪══════════════════════╪════════════════╡
│ ENSG00000020219.9 ┆ CCT8L1P   ┆ processed_pseudogene ┆ 0.0            │
│ ENSG00000082929.8 ┆ LINC01587 ┆ lncRNA               ┆ 0.2336         │
│ ENSG00000194297.2 ┆ RNU1-75P  ┆ snRNA                ┆ 0.0            │
│ ENSG00000194717.4 ┆ MIR494    ┆ miRNA                ┆ 0.0            │
└───────────────────┴───────────┴──────────────────────┴────────────────┘


In [13]:
# Nasze śledzone geny - czy przetrwały filtr?
print("=== Śledzone geny po filtrze biotype ===\n")
for ensg, desc in TRACKED_GENES.items():
    in_kept = kept.filter(pl.col("gene_id").str.starts_with(ensg)).height > 0
    status = "✓ ZACHOWANY" if in_kept else "✗ odrzucony"
    print(f"  {status}  {desc}")

print(f"\nWszystkie markery LUAD to protein_coding -> przetrwały filtr.")

=== Śledzone geny po filtrze biotype ===

  ✓ ZACHOWANY  NKX2-1 (marker różnicowania, protein_coding)
  ✓ ZACHOWANY  EGFR (onkogen, protein_coding)
  ✓ ZACHOWANY  KRAS (onkogen, protein_coding)

Wszystkie markery LUAD to protein_coding -> przetrwały filtr.


**Co się zmieniło na Etapie 3:**

- Z 60660 genów zostało **19962** (`protein_coding`)
- Odrzucone: lncRNA (16901), pseudogeny (~14000), miRNA, snRNA itd.
- Nasze markery (NKX2-1, EGFR, KRAS) to geny kodujące białka → **wszystkie zachowane**

Liczba 19962 to dokładnie wymiar finalnej macierzy w pipeline (zgodny z notebookami 02, 05, 06).

---
## Etap 4 — Finalna macierz (pełna kohorta)

Powyższe etapy pokazaliśmy na **jednym pliku**. W praktyce `build_expression_matrix` robi to dla **wszystkich 601 plików** kohorty naraz, składając je w jedną macierz:

- każdy plik → jedna kolumna (nazwana `sample_id`)
- wiersze = geny (wspólne dla wszystkich plików, ta sama kolejność)
- po filtrze biotype: 19962 genów × N próbek

Wczytujemy **prawdziwą macierz** zbudowaną przez pipeline, by zobaczyć gdzie wylądował nasz plik:

In [14]:
# Wczytanie finalnej macierzy z pipeline (jeśli istnieje)
matrix_path = PROJECT_ROOT / "data" / "processed" / "expression_matrix.parquet"

if matrix_path.exists():
    matrix = pl.read_parquet(matrix_path)
    n_genes = matrix.height
    n_samples = matrix.width - 1  # -1 bo gene_id
    print(f"=== Finalna macierz ekspresji ===")
    print(f"Wymiary: {n_genes} genów × {n_samples} próbek")
    print(f"Pierwsza kolumna: {matrix.columns[0]}")
    print(f"Przykładowe sample_id (kolumny): {matrix.columns[1:4]}")
    print()

    # NKX2-1 w pełnej macierzy - wartości w pierwszych próbkach
    nkx = matrix.filter(pl.col("gene_id").str.starts_with("ENSG00000136352"))
    if nkx.height > 0:
        print("=== NKX2-1 w finalnej macierzy (pierwsze 5 próbek) ===")
        cols_show = ["gene_id"] + matrix.columns[1:6]
        print(nkx.select(cols_show))
else:
    print("Macierz nie istnieje jeszcze w data/processed/.")
    print("Zbuduj ją przez: uv run python -m src.cli build-matrix --config configs/default.yaml --duplicate-strategy deepest")

=== Finalna macierz ekspresji ===
Wymiary: 19962 genów × 590 próbek
Pierwsza kolumna: gene_id
Przykładowe sample_id (kolumny): ['TCGA-44-2666-01B', 'TCGA-69-A59K-01A', 'TCGA-99-8032-01A']

=== NKX2-1 w finalnej macierzy (pierwsze 5 próbek) ===
shape: (1, 6)
┌────────────────┬────────────────┬────────────────┬───────────────┬───────────────┬───────────────┐
│ gene_id        ┆ TCGA-44-2666-0 ┆ TCGA-69-A59K-0 ┆ TCGA-99-8032- ┆ TCGA-95-7562- ┆ TCGA-91-6848- │
│ ---            ┆ 1B             ┆ 1A             ┆ 01A           ┆ 01A           ┆ 01A           │
│ str            ┆ ---            ┆ ---            ┆ ---           ┆ ---           ┆ ---           │
│                ┆ f64            ┆ f64            ┆ f64           ┆ f64           ┆ f64           │
╞════════════════╪════════════════╪════════════════╪═══════════════╪═══════════════╪═══════════════╡
│ ENSG0000013635 ┆ 115.4209       ┆ 222.6885       ┆ 193.4675      ┆ 187.7981      ┆ 4.8766        │
│ 2.19           ┆                ┆

---
## Podsumowanie podróży

Prześledziliśmy jeden plik STAR-Counts przez cały pipeline:

| Etap | Funkcja | Wymiary | Co się zmieniło |
|---|---|---|---|
| **0. Surowy** | — | ~60666 linii × 9 kol | komentarz #, nagłówek, 4 meta, geny; pola tekstowe |
| **1. Parse** | `parse_star_counts` | 60660 × 9 | usunięte komentarz/meta, zliczenia → Int64, walidacja |
| **2. Metryka** | `build_expression_matrix` | 60660 × 2 | z 9 kolumn → gene_id + tpm_unstranded |
| **3. Biotype** | `build_expression_matrix` | 19962 × 2 | tylko protein_coding (z 60660) |
| **4. Macierz** | `build_expression_matrix` | 19962 × N | plik = jedna kolumna (sample_id) wśród 601 |

**Kluczowa obserwacja:** plik na wejściu (4 MB tekstu z metadanymi, statystykami, sześcioma metrykami dla 60660 genów) staje się na końcu **jedną kolumną liczb TPM dla 19962 genów kodujących białka** — gotową do analizy ekspresji i przeżywalności.

Każda transformacja jest **deterministyczna i jawna**: pipeline stosuje zdefiniowane reguły (które kolumny, które wiersze, który biotyp). To czyni przetwarzanie odtwarzalnym — ten sam plik zawsze da tę samą kolumnę macierzy.